# Практика 9. Преобразование, объединение и агрегирование данных

На практике 8 мы привели данные в порядок. Теперь из них нужно получить ответы: кто из
перевозчиков работает лучше, какие маршруты приносят выручку, выполнен ли план.

Для этого данных одной таблицы не хватает. Рейсы лежат отдельно за апрель и май, рейтинги
перевозчиков — в справочнике, сегменты клиентов — в другом, план — в третьем. Сегодня
учимся собирать это вместе и сворачивать в сводные таблицы.

| Часть | О чём |
|---|---|
| 1 | `concat` — склейка таблиц одинаковой структуры |
| 2 | Сортировка, фильтрация, преобразование таблиц |
| 3 | `groupby` и агрегирующие функции |
| 4 | `merge` — соединение по ключу |
| 5 | `pivot_table` и сводные представления |
| 6 | Мини-проект: план/факт по маршрутам |

> **Главная мысль пары:** соединение таблиц — самая опасная операция в аналитике.
> Она умеет **молча менять число строк**, и тогда все суммы становятся неверными.
> В части 4 вы увидите, как одна лишняя строка в справочнике завышает выручку на 3.7 млн
> рублей. Поэтому правило: после каждого соединения — проверка числа строк.

---
## Шаг 0. Проект

```
# терминал
mkdir pandas-aggregation
cd pandas-aggregation
```

`requirements.txt`:

```
pandas
```

```
# терминал (Windows)
python -m venv .venv
.\.venv\Scripts\Activate.ps1
pip install -r requirements.txt
```

```
# терминал (macOS / Linux)
python3 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
```

---
## Данные

Запустите ячейку — она создаст пять таблиц. Это типичная ситуация: данные о событиях
отдельно, справочники отдельно, план отдельно.

| Файл | Что внутри | Зачем |
|---|---|---|
| `trips_april.csv` | рейсы за апрель | склеиваем с майскими |
| `trips_may.csv` | рейсы за май | склеиваем с апрельскими |
| `carriers.csv` | перевозчики: город, парк машин, рейтинг | присоединяем к рейсам |
| `clients.csv` | клиенты: сегмент, менеджер, год начала работы | присоединяем к рейсам |
| `plan.csv` | план по маршрутам на май | сравниваем с фактом |

In [56]:
# ЯЧЕЙКА-ГЕНЕРАТОР ДАННЫХ. Запустите её (Shift+Enter), менять ничего не нужно.
# Создаёт пять таблиц: рейсы за апрель и май, справочники перевозчиков и клиентов, план по маршрутам.

import csv
import random
from pathlib import Path

random.seed(9)
Path("data").mkdir(exist_ok=True)

routes = {
    "Москва-Казань": 820, "Москва-Санкт-Петербург": 710, "Москва-Нижний Новгород": 420,
    "Москва-Воронеж": 520, "Казань-Самара": 360, "Екатеринбург-Челябинск": 220,
    "Новосибирск-Омск": 680, "Санкт-Петербург-Петрозаводск": 430,
}
carriers = ["ТрансЛогистик", "РусКарго", "СибПуть", "ВолгаТранс", "АвтоДорСервис"]
tariffs = {"Контейнер": 7.5, "Нефтепродукты": 9.2, "Зерно": 5.1,
           "Металл": 6.8, "Лес": 4.6, "Стройматериалы": 5.9}
clients = ["Агрохолдинг Нива", "МеталлПром", "СтройБаза 77", "НефтеТрейд", "ЛесЭкспорт", "РитейлСеть"]


def make_trips(n, month, start_id):
    rows = []
    for i in range(start_id, start_id + n):
        route, distance = random.choice(list(routes.items()))
        cargo = random.choice(list(tariffs))
        weight = round(random.uniform(3, 22), 1)
        delay = 0 if random.random() < 0.72 else random.choice([1, 2, 3, 4, 6, 8])
        rows.append({
            "trip_id": f"T{i:04d}",
            "date": f"2025-{month:02d}-{random.randint(1, 28):02d}",
            "route": route,
            "carrier": random.choice(carriers),
            "cargo_type": cargo,
            "client": random.choice(clients),
            "weight_t": weight,
            "distance_km": distance,
            "cost_rub": int(round(distance * weight * tariffs[cargo] * random.uniform(0.95, 1.08), -1)),
            "delay_h": delay,
        })
    return rows


def save(path, rows, fields=None):
    with open(path, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fields or list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)


april = make_trips(250, 4, 1)
may = make_trips(280, 5, 1001)

# в майской выгрузке один рейс приписан перевозчику, которого нет в справочнике
may[7]["carrier"] = "НовыйПеревозчик"

save("data/trips_april.csv", april)
save("data/trips_may.csv", may)

# Справочник перевозчиков. Внимание: одна строка случайно задвоена —
# это пригодится, когда будем проверять соединение таблиц.
carriers_rows = [
    {"carrier": "ТрансЛогистик", "city": "Москва", "fleet": 48, "rating": 4.6},
    {"carrier": "РусКарго", "city": "Москва", "fleet": 35, "rating": 4.1},
    {"carrier": "СибПуть", "city": "Новосибирск", "fleet": 22, "rating": 3.8},
    {"carrier": "ВолгаТранс", "city": "Казань", "fleet": 29, "rating": 4.4},
    {"carrier": "АвтоДорСервис", "city": "Воронеж", "fleet": 18, "rating": 3.5},
    {"carrier": "ВолгаТранс", "city": "Казань", "fleet": 29, "rating": 4.4},
]
save("data/carriers.csv", carriers_rows)

# Справочник клиентов: есть клиент, который в рейсах не встречается
clients_rows = [
    {"client": "Агрохолдинг Нива", "segment": "Агро", "manager": "Петрова", "since": 2019},
    {"client": "МеталлПром", "segment": "Промышленность", "manager": "Иванов", "since": 2017},
    {"client": "СтройБаза 77", "segment": "Строительство", "manager": "Иванов", "since": 2021},
    {"client": "НефтеТрейд", "segment": "Энергетика", "manager": "Соколов", "since": 2018},
    {"client": "ЛесЭкспорт", "segment": "Лес", "manager": "Петрова", "since": 2020},
    {"client": "РитейлСеть", "segment": "Ритейл", "manager": "Соколов", "since": 2022},
    {"client": "АгроЭкспорт Юг", "segment": "Агро", "manager": "Петрова", "since": 2024},
]
save("data/clients.csv", clients_rows)

# План по маршрутам на май: ставился до начала месяца, поэтому близок к факту, но не равен ему
fact_by_route = {}
for row in may:
    key = row["route"]
    agg = fact_by_route.setdefault(key, {"trips": 0, "revenue": 0})
    agg["trips"] += 1
    agg["revenue"] += row["cost_rub"]

plan_rows = []
for route in routes:
    agg = fact_by_route.get(route, {"trips": 30, "revenue": 1500000})
    plan_rows.append({
        "route": route,
        "month": "2025-05",
        "plan_trips": max(1, round(agg["trips"] * random.uniform(0.85, 1.2))),
        "plan_revenue": int(round(agg["revenue"] * random.uniform(0.85, 1.2), -3)),
    })
save("data/plan.csv", plan_rows)

print("Готово. Создано:")
print("  data/trips_april.csv — рейсы за апрель:", len(april), "строк")
print("  data/trips_may.csv   — рейсы за май:   ", len(may), "строк")
print("  data/carriers.csv    — справочник перевозчиков:", len(carriers_rows), "строк")
print("  data/clients.csv     — справочник клиентов:    ", len(clients_rows), "строк")
print("  data/plan.csv        — план по маршрутам:      ", len(plan_rows), "строк")

Готово. Создано:
  data/trips_april.csv — рейсы за апрель: 250 строк
  data/trips_may.csv   — рейсы за май:    280 строк
  data/carriers.csv    — справочник перевозчиков: 6 строк
  data/clients.csv     — справочник клиентов:     7 строк
  data/plan.csv        — план по маршрутам:       8 строк


---
# Часть 1. `concat` — склейка таблиц

`concat` ставит таблицы **друг под друга**. Применяется, когда структура одинаковая,
а данные относятся к разным периодам или источникам: месяц к месяцу, филиал к филиалу.

Два обязательных параметра на практике:

| Параметр | Зачем |
|---|---|
| `ignore_index=True` | сквозная нумерация: иначе индексы повторятся (0,1,2,... 0,1,2,...) |
| `keys=[...]` | пометить, откуда пришла строка |

И обязательная проверка после склейки: **сумма строк исходных таблиц должна совпасть
с числом строк результата**.

In [57]:
import pandas as pd

april = pd.read_csv("data/trips_april.csv", parse_dates=["date"])
may = pd.read_csv("data/trips_may.csv", parse_dates=["date"])

print("апрель:", april.shape, "| май:", may.shape)
print("столбцы совпадают:", list(april.columns) == list(may.columns))
print()

trips = pd.concat([april, may], ignore_index=True)

print("после склейки:", trips.shape)
print("проверка:", len(april), "+", len(may), "=", len(trips),
      "->", "OK" if len(april) + len(may) == len(trips) else "ОШИБКА")
print("дубликатов trip_id:", trips["trip_id"].duplicated().sum())

апрель: (250, 10) | май: (280, 10)
столбцы совпадают: True

после склейки: (530, 10)
проверка: 250 + 280 = 530 -> OK
дубликатов trip_id: 0


In [58]:
# Удобный признак месяца — он понадобится во всех группировках
trips["month"] = trips["date"].dt.to_period("M").astype(str)
print(trips["month"].value_counts().sort_index())
print()

# Если нужно помнить источник строки — параметр keys
marked = pd.concat([april, may], keys=["апрель", "май"], names=["источник", "строка"])
print(marked.index[:3].tolist())

month
2025-04    250
2025-05    280
Name: count, dtype: int64

[('апрель', 0), ('апрель', 1), ('апрель', 2)]


> **Что будет, если структура таблиц не совпадает.** `concat` не упадёт: он объединит
> все столбцы, а отсутствующие заполнит `NaN`. Выглядит как «всё получилось», а на деле
> у половины строк пустой столбец. Поэтому сравнение `list(a.columns) == list(b.columns)`
> перед склейкой — не формальность.

### Задание 1

1. Склейте апрель и май в таблицу `trips` со сквозной нумерацией. Проверьте число строк.
2. Добавьте столбец `month`.
3. Проверьте, что номера рейсов не повторяются. Почему это важно проверить именно
   **после** склейки?
4. Сымитируйте проблему: удалите из копии майской таблицы столбец `delay_h`
   (`may.drop(columns=["delay_h"])`), склейте с апрелем и посмотрите, сколько получилось
   пропусков в `delay_h`. Какой вывод?

In [59]:
# 1)
trips = pd.concat([april, may], ignore_index=True)
print("строк:", len(april) + len(may) == len(trips))

# 2)
trips["month"] = trips["date"].dt.month

# 3)
print("дубликатов trip_id:", trips["trip_id"].duplicated().sum())
# потому что в таблицах могут быть одинаковые строчки и их необходимо будет удалить

# 4)
broken = pd.concat([april, may.drop(columns=["delay_h"])], ignore_index=True)
print("пропусков в delay_h:", broken["delay_h"].isna().sum()) 
#пропусков получилось ровно столько, сколько было строк в таблице с маем)))

строк: True
дубликатов trip_id: 0
пропусков в delay_h: 280


---
# Часть 2. Сортировка, фильтрация, преобразование

Базовые операции, которые понадобятся в каждой задаче.

| Задача | Код |
|---|---|
| сортировка по одному столбцу | `df.sort_values("cost_rub", ascending=False)` |
| сортировка по нескольким | `df.sort_values(["route", "cost_rub"], ascending=[True, False])` |
| топ-N без сортировки всей таблицы | `df.nlargest(10, "cost_rub")` |
| фильтр | `df[(df["a"] > 1) & (df["b"] < 2)]` |
| фильтр строкой | `df.query("distance_km > 500 and delay_h > 0")` |
| новый столбец в цепочке | `df.assign(cost_per_km=lambda d: d["cost_rub"] / d["distance_km"])` |
| переименовать / удалить | `df.rename(columns={...})`, `df.drop(columns=[...])` |

`query` и `assign` удобны тем, что их можно выстроить в одну цепочку — код читается
сверху вниз как список действий.

In [60]:
# Топ-3 самых дорогих рейсов
print(trips.nlargest(3, "cost_rub")[["trip_id", "route", "cargo_type", "cost_rub"]])
print()

# Сортировка по двум столбцам: маршрут по алфавиту, внутри — по убыванию стоимости
print(trips.sort_values(["route", "cost_rub"], ascending=[True, False])
           .head(4)[["route", "trip_id", "cost_rub"]])

    trip_id                   route     cargo_type  cost_rub
394   T1145           Москва-Казань  Нефтепродукты    167180
62    T0063  Москва-Санкт-Петербург  Нефтепродукты    125500
277   T1028  Москва-Санкт-Петербург      Контейнер    123600

                      route trip_id  cost_rub
284  Екатеринбург-Челябинск   T1035     46960
27   Екатеринбург-Челябинск   T0028     40450
282  Екатеринбург-Челябинск   T1033     38870
114  Екатеринбург-Челябинск   T0115     35550


In [61]:
# Один и тот же фильтр двумя способами
mask_way = trips[(trips["distance_km"] > 500) & (trips["delay_h"] > 0)]
query_way = trips.query("distance_km > 500 and delay_h > 0")

print("маской:", len(mask_way), "| query:", len(query_way))
print()

# Цепочка: фильтр -> новый признак -> сортировка -> нужные столбцы
result = (trips
          .query("month == '2025-05'")
          .assign(cost_per_km=lambda d: (d["cost_rub"] / d["distance_km"]).round(2))
          .sort_values("cost_per_km", ascending=False)
          [["trip_id", "route", "cargo_type", "cost_per_km"]]
          .head(5))
print(result)

маской: 63 | query: 63

Empty DataFrame
Columns: [trip_id, route, cargo_type, cost_per_km]
Index: []


### Задание 2

1. Выведите 10 самых дорогих рейсов за май — только `trip_id`, `route`, `cargo_type`, `cost_rub`.
2. Отсортируйте рейсы по перевозчику (по алфавиту), внутри каждого — по убыванию стоимости.
   Выведите первые 10 строк.
3. Сколько рейсов с грузом `Нефтепродукты` на маршрутах длиннее 600 км? Сделайте двумя
   способами — маской и через `query` — и сверьте числа.
4. Одной цепочкой: возьмите апрельские рейсы, добавьте признак `cost_per_ton`
   (стоимость за тонну), оставьте рейсы дороже 3000 руб. за тонну, отсортируйте по убыванию.
   Сколько таких рейсов?

In [62]:
# 1)
print(trips[trips["month"] == 5].nlargest(10, "cost_rub")[["trip_id", "route", "cargo_type", "cost_rub"]])

# 2)
print(trips.sort_values(["carrier", "cost_rub"], ascending=[True, False]).head(10))

# 3)
print("маской:", len(trips[(trips["cargo_type"] == "Нефтепродукты") & (trips["distance_km"] > 600)]), "| query:", len(trips.query('distance_km > 600 and cargo_type == "Нефтепродукты"')))

# 4)
expensive = (
    trips
    .query("month == 4")
    .assign(cost_per_ton=lambda d: d["cost_rub"] / d["weight_t"])
    .query("cost_per_ton > 3000")
    .sort_values("cost_per_ton", ascending=False)
)
print(len(expensive))

    trip_id                   route      cargo_type  cost_rub
394   T1145           Москва-Казань   Нефтепродукты    167180
277   T1028  Москва-Санкт-Петербург       Контейнер    123600
408   T1159           Москва-Казань          Металл    121420
395   T1146  Москва-Санкт-Петербург       Контейнер    113860
311   T1062  Москва-Санкт-Петербург   Нефтепродукты    107860
425   T1176           Москва-Казань       Контейнер     97940
257   T1008  Москва-Санкт-Петербург   Нефтепродукты     97050
363   T1114          Москва-Воронеж   Нефтепродукты     95610
481   T1232           Москва-Казань       Контейнер     93520
497   T1248           Москва-Казань  Стройматериалы     91550
    trip_id       date                   route        carrier     cargo_type  \
394   T1145 2025-05-20           Москва-Казань  АвтоДорСервис  Нефтепродукты   
42    T0043 2025-04-01        Новосибирск-Омск  АвтоДорСервис  Нефтепродукты   
173   T0174 2025-04-09  Москва-Санкт-Петербург  АвтоДорСервис         Металл  

---
# Часть 3. `groupby` и агрегирующие функции

`groupby` — главный инструмент аналитика. Он делает то, что на практике 2 мы писали
вручную циклом со словарём-накопителем: разбивает таблицу на группы и считает по каждой.

**Схема работы:** разбить по ключу → применить функцию → собрать результат.

| Запись | Что делает |
|---|---|
| `df.groupby("route")["cost_rub"].sum()` | одна функция к одному столбцу |
| `df.groupby("route")["cost_rub"].agg(["sum", "mean", "count"])` | несколько функций |
| `df.groupby(["month", "carrier"])[...]` | группировка по двум ключам |
| `df.groupby("route").agg(revenue=("cost_rub", "sum"))` | **именованная агрегация** — лучший способ |
| `df.groupby("route").size()` | число строк в группе |
| `.reset_index()` | вернуть ключ группировки из индекса в столбец |

Именованная агрегация читается лучше всего: сразу видно, как будет называться столбец
и из чего он считается.

In [63]:
# Простейший случай
print(trips.groupby("route")["cost_rub"].sum().sort_values(ascending=False))
print()

# Несколько показателей сразу — именованная агрегация
by_route = (trips.groupby("route")
            .agg(trips_count=("trip_id", "count"),
                 revenue=("cost_rub", "sum"),
                 avg_weight=("weight_t", "mean"),
                 max_cost=("cost_rub", "max"))
            .round(2)
            .sort_values("revenue", ascending=False))
print(by_route)

route
Москва-Санкт-Петербург          4636630
Москва-Казань                   4141680
Новосибирск-Омск                2710710
Москва-Нижний Новгород          2589090
Москва-Воронеж                  2458350
Санкт-Петербург-Петрозаводск    2097460
Казань-Самара                   1890780
Екатеринбург-Челябинск          1402900
Name: cost_rub, dtype: int64

                              trips_count  revenue  avg_weight  max_cost
route                                                                   
Москва-Санкт-Петербург                 74  4636630       13.65    125500
Москва-Казань                          69  4141680       11.82    167180
Новосибирск-Омск                       50  2710710       11.65    109740
Москва-Нижний Новгород                 74  2589090       12.90     73350
Москва-Воронеж                         60  2458350       11.66     99370
Санкт-Петербург-Петрозаводск           63  2097460       11.41     82840
Казань-Самара                          66  1890780       11.

In [64]:
# Своя функция внутри агрегации: доля рейсов с задержкой
by_carrier = (trips.groupby("carrier")
              .agg(trips_count=("trip_id", "count"),
                   revenue=("cost_rub", "sum"),
                   delayed_share=("delay_h", lambda s: (s > 0).mean()))
              .round(3)
              .sort_values("revenue", ascending=False))
print(by_carrier)
print()

# Группировка по двум ключам даёт двухуровневый индекс
by_two = trips.groupby(["month", "carrier"])["cost_rub"].sum()
print(by_two.head(6))
print()
print("тот же результат в виде обычной таблицы:")
print(by_two.reset_index().head(3))

                 trips_count  revenue  delayed_share
carrier                                             
ТрансЛогистик            120  5028610          0.150
АвтоДорСервис            119  4519180          0.261
СибПуть                  102  4477860          0.294
РусКарго                  96  4096230          0.188
ВолгаТранс                92  3708670          0.402
НовыйПеревозчик            1    97050          0.000

month  carrier      
4      АвтоДорСервис    2045080
       ВолгаТранс       1721460
       РусКарго         2512120
       СибПуть          1950630
       ТрансЛогистик    2367060
5      АвтоДорСервис    2474100
Name: cost_rub, dtype: int64

тот же результат в виде обычной таблицы:
   month        carrier  cost_rub
0      4  АвтоДорСервис   2045080
1      4     ВолгаТранс   1721460
2      4       РусКарго   2512120


## `size` против `count`

Разница незаметна, пока в данных нет пропусков, и становится важной, когда они есть:

- `size()` — сколько **строк** в группе;
- `count()` — сколько **непустых значений** в конкретном столбце.

## `transform` — вернуть результат в исходную таблицу

`agg` сворачивает группу в одну строку. `transform` считает по группе, но возвращает
значение **для каждой строки** — удобно, когда нужна доля или сравнение со средним по группе.

In [65]:
print("size (строк в группе):")
print(trips.groupby("carrier").size())
print()

# transform: какую долю рейс занимает в выручке своего маршрута
trips["route_revenue"] = trips.groupby("route")["cost_rub"].transform("sum")
trips["share_in_route"] = (trips["cost_rub"] / trips["route_revenue"] * 100).round(2)

print(trips[["trip_id", "route", "cost_rub", "route_revenue", "share_in_route"]].head(5))
print()
print("сумма долей по одному маршруту:",
      trips[trips["route"] == "Москва-Казань"]["share_in_route"].sum().round(1), "%")

size (строк в группе):
carrier
АвтоДорСервис      119
ВолгаТранс          92
НовыйПеревозчик      1
РусКарго            96
СибПуть            102
ТрансЛогистик      120
dtype: int64

  trip_id                         route  cost_rub  route_revenue  \
0   T0001  Санкт-Петербург-Петрозаводск     20280        2097460   
1   T0002        Москва-Санкт-Петербург     50910        4636630   
2   T0003  Санкт-Петербург-Петрозаводск     28350        2097460   
3   T0004        Москва-Санкт-Петербург     42250        4636630   
4   T0005              Новосибирск-Омск     88100        2710710   

   share_in_route  
0            0.97  
1            1.10  
2            1.35  
3            0.91  
4            3.25  

сумма долей по одному маршруту: 100.0 %


### Задание 3

1. Посчитайте по каждому **перевозчику**: число рейсов, суммарную выручку, средний вес,
   среднюю задержку и долю рейсов с задержкой. Используйте именованную агрегацию.
   Отсортируйте по выручке.
2. Посчитайте выручку по парам «месяц — тип груза». Приведите результат к обычной таблице
   через `reset_index()`.
3. У какого перевозчика самая высокая **средняя стоимость рейса**? А у какого — самая
   высокая **суммарная**? Это один и тот же перевозчик? Объясните результат.
4. Через `transform` добавьте столбец `vs_route_avg` — во сколько раз стоимость рейса
   отличается от средней по его маршруту. Найдите рейс с максимальным значением.
5. Сравните `groupby("client").size()` и `groupby("client")["delay_h"].count()`.
   Совпали? Почему?

In [66]:
# 1)
by_carrier = (
    trips.groupby("carrier")
    .agg(
        trips_n=("trip_id", "count"),
        revenue=("cost_rub", "sum"),
        avg_weight=("weight_t", "mean"),
        avg_delay=("delay_h", "mean"),
        delayed_share=("delay_h", lambda s: (s > 0).mean()),
    )
    .sort_values("revenue", ascending=False)
)
print(by_carrier)
print()

# 2)
by_month_cargo = (
    trips.groupby(["month", "cargo_type"])["cost_rub"]
    .sum()
    .reset_index()
)
print(by_month_cargo)
print()

# 3)
stats = trips.groupby("carrier")["cost_rub"].agg(["mean", "sum", "count"])
print("макс. средняя - ", stats["mean"].idxmax())
print("макс. суммарная -", stats["sum"].idxmax())
print('Это один и тот же перевозчик?', stats['mean'].idxmax()==stats['sum'].idxmax())
#перевозчик может возить мало, но дорогих рейсов (высокая средняя) или много обычных (высокая сумма). 
#если это разные перевозчики, значит, лидер по макс. средней берёт объёмом, а не ценой.
print()


# 4)
trips["vs_route_avg"] = trips["cost_rub"] / trips.groupby("route")["cost_rub"].transform("mean")
print(trips.loc[trips["vs_route_avg"].idxmax()])
print()

# 5)
print(trips.groupby("client").size() == trips.groupby("client")["delay_h"].count())
#они совпали потому что в delay_h нет пропусков.

                 trips_n  revenue  avg_weight  avg_delay  delayed_share
carrier                                                                
ТрансЛогистик        120  5028610   12.745833   0.600000       0.150000
АвтоДорСервис        119  4519180   11.457983   0.957983       0.260504
СибПуть              102  4477860   12.916667   1.039216       0.294118
РусКарго              96  4096230   12.568750   0.583333       0.187500
ВолгаТранс            92  3708670   11.690217   1.456522       0.402174
НовыйПеревозчик        1    97050   15.000000   0.000000       0.000000

    month      cargo_type  cost_rub
0       4           Зерно   1044390
1       4       Контейнер   1913370
2       4             Лес   1307210
3       4          Металл   2096660
4       4   Нефтепродукты   2332100
5       4  Стройматериалы   1902620
6       5           Зерно   1598130
7       5       Контейнер   1882730
8       5             Лес   1418600
9       5          Металл   2009360
10      5   Нефтепродукты  

---
# Часть 4. `merge` — соединение по ключу

`merge` присоединяет к таблице столбцы **из другой таблицы** по общему ключу.
Это аналог `VLOOKUP` в Excel и `JOIN` в SQL.

Четыре типа соединения — параметр `how`:

| `how` | Что остаётся в результате |
|---|---|
| `"left"` | все строки левой таблицы; не нашлось справа — `NaN` |
| `"inner"` | только те, что нашлись в обеих |
| `"right"` | все строки правой |
| `"outer"` | все строки из обеих |

**Для аналитики по умолчанию — `how="left"`.** Левая таблица это ваши факты (рейсы),
и терять их нельзя: если перевозчика нет в справочнике, рейс всё равно был.

Два параметра, которые спасают от тихих ошибок:

- `indicator=True` — добавляет столбец с пометкой, откуда взялась строка;
- `validate="m:1"` — проверка «много к одному»: падает с ошибкой, если ключ в правой
  таблице не уникален.

In [67]:
carriers = pd.read_csv("data/carriers.csv")
clients = pd.read_csv("data/clients.csv")

print("справочник перевозчиков:")
print(carriers)
print()
print("строк:", len(carriers), "| уникальных перевозчиков:", carriers["carrier"].nunique())

справочник перевозчиков:
         carrier         city  fleet  rating
0  ТрансЛогистик       Москва     48     4.6
1       РусКарго       Москва     35     4.1
2        СибПуть  Новосибирск     22     3.8
3     ВолгаТранс       Казань     29     4.4
4  АвтоДорСервис      Воронеж     18     3.5
5     ВолгаТранс       Казань     29     4.4

строк: 6 | уникальных перевозчиков: 5


Посмотрите на последнюю строку вывода внимательно: строк шесть, а перевозчиков пять.
Запомните это и запустите следующую ячейку.

In [68]:
# Присоединяем справочник "как есть"
merged = trips.merge(carriers, on="carrier", how="left")

print("строк в рейсах было: ", len(trips))
print("строк после merge:   ", len(merged))
print("прибавилось:         ", len(merged) - len(trips))
print()
print(f"выручка до merge:    {trips['cost_rub'].sum():>12,}".replace(",", " "))
print(f"выручка после merge: {merged['cost_rub'].sum():>12,}".replace(",", " "))
print(f"завышение:           {merged['cost_rub'].sum() - trips['cost_rub'].sum():>12,}".replace(",", " "))

строк в рейсах было:  530
строк после merge:    622
прибавилось:          92

выручка до merge:      21 927 600
выручка после merge:   25 636 270
завышение:              3 708 670


Выручка выросла на 3.7 миллиона, хотя мы всего лишь присоединили справочник с рейтингами.
**Ни одной ошибки не было.**

Причина: в справочнике перевозчик `ВолгаТранс` записан дважды. Для каждого рейса этого
перевозчика `merge` нашёл **два** подходящих значения справа — и создал две строки вместо
одной. Все 92 рейса `ВолгаТранс` задвоились, и их стоимость попала в сумму дважды.

Это самая частая и самая дорогая ошибка при соединении таблиц. Защита — две строки:

In [69]:
# 1. Проверка "много к одному": падает сразу, если ключ справа не уникален
try:
    trips.merge(carriers, on="carrier", how="left", validate="m:1")
except Exception as e:
    print(type(e).__name__, "->", e)
print()

# 2. Лечение: убрать дубли в справочнике
carriers_ok = carriers.drop_duplicates(subset="carrier")
merged = trips.merge(carriers_ok, on="carrier", how="left", validate="m:1")

print("строк после merge:", len(merged), "| было:", len(trips),
      "->", "OK" if len(merged) == len(trips) else "ОШИБКА")
print(f"выручка: {merged['cost_rub'].sum():,}".replace(",", " "), "- совпадает с исходной")

MergeError -> Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
    carrier
ВолгаТранс ...

строк после merge: 530 | было: 530 -> OK
выручка: 21 927 600 - совпадает с исходной


In [70]:
# Кто не нашёлся в справочнике
print("рейсов без данных о перевозчике:", merged["rating"].isna().sum())
print("это перевозчик:", merged.loc[merged["rating"].isna(), "carrier"].unique().tolist())
print()

# indicator показывает, с какой стороны пришла каждая строка
check = trips.merge(clients, on="client", how="outer", indicator=True)
print(check["_merge"].value_counts())
print()
print("есть в справочнике, но рейсов не было:",
      check.loc[check["_merge"] == "right_only", "client"].unique().tolist())

рейсов без данных о перевозчике: 1
это перевозчик: ['НовыйПеревозчик']

_merge
both          530
right_only      1
left_only       0
Name: count, dtype: int64

есть в справочнике, но рейсов не было: ['АгроЭкспорт Юг']


Два вида несовпадений, и оба нужно замечать:

- **`left_only`** — факт есть, справочника нет. У нас это рейс перевозчика
  `НовыйПеревозчик`: рейтинга мы не знаем, но рейс состоялся и выручку принёс.
  Выбрасывать нельзя.
- **`right_only`** — справочник есть, фактов нет. Клиент `АгроЭкспорт Юг` в договоре
  числится, но ни одного рейса за два месяца. Для отчёта по выручке он не нужен,
  а вот менеджеру по работе с клиентами это интересно.

### Задание 4

1. Присоедините к `trips` справочник перевозчиков — правильно, с проверкой числа строк.
2. Присоедините справочник клиентов (`how="left"`). Проверьте число строк снова.
3. Сколько рейсов осталось без `segment`? Почему?
4. Сделайте то же соединение с `how="inner"` и сравните число строк с `how="left"`.
   Какие рейсы потерялись и почему это плохо для отчёта о выручке?
5. Посчитайте выручку по сегментам клиентов. Какой сегмент крупнейший?

In [75]:
# 1)
carriers_clean = carriers.drop_duplicates(subset="carrier")
full = trips.merge(carriers_clean, on='carrier', how='left', validate='m:1')
print("строк:", len(full), "| было:", len(trips), "->", "OK" if len(full) == len(trips) else "ОШИБКА")

# 2)
full = full.merge(clients, on="client", how="left", validate="m:1", indicator=True)
print("строк:", len(full), "->", "OK" if len(full) == len(trips) else "ОШИБКА")

# 3)
print("рейсов без сегмента:", full["segment"].isna().sum())

# 4)
inner = trips.merge(clients, on="client", how="inner", validate="m:1")
print("left:", len(full), "| inner:", len(inner))
#у меня ничего не потерялось... опасно потому что этот рейс может быть не учтен, а соответственно отчет будет неверный. 

# 5)
by_segment = (full.groupby("segment", dropna=False)["cost_rub"].sum().sort_values(ascending=False))
print('крупнейший - ', by_segment.idxmax())

строк: 530 | было: 530 -> OK
строк: 530 -> OK
рейсов без сегмента: 0
left: 530 | inner: 530
крупнейший -  Промышленность


---
# Часть 5. `pivot_table` и сводные представления

Сводная таблица — это группировка по **двум** ключам, разложенная в прямоугольник:
один ключ в строки, другой в столбцы. Такой вид удобно читать глазами и показывать
на совещании.

```python
pd.pivot_table(df,
               index="route",        # что в строках
               columns="cargo_type", # что в столбцах
               values="cost_rub",    # что считаем
               aggfunc="sum",        # как считаем
               fill_value=0,         # чем заполнить пустые ячейки
               margins=True)         # добавить строку и столбец "Итого"
```

| Инструмент | Когда |
|---|---|
| `pivot_table` | нужны числа по двум измерениям с агрегацией |
| `crosstab` | нужно просто посчитать количество пересечений |
| `pivot` | данные уже агрегированы, нужно только «развернуть» |

In [76]:
full = trips.merge(carriers_ok, on="carrier", how="left", validate="m:1")
full = full.merge(clients, on="client", how="left")

pivot = pd.pivot_table(full,
                       index="route",
                       columns="cargo_type",
                       values="cost_rub",
                       aggfunc="sum",
                       fill_value=0,
                       margins=True,
                       margins_name="Итого")

print("Выручка по маршрутам и типам груза:")
print(pivot.iloc[:4, :4])
print()
print("общая выручка (угловая ячейка):", f"{pivot.loc['Итого', 'Итого']:,}".replace(",", " "))

Выручка по маршрутам и типам груза:
cargo_type               Зерно  Контейнер     Лес  Металл
route                                                    
Екатеринбург-Челябинск  194240     252960  143890  223640
Казань-Самара           238360     385450  226830  286560
Москва-Воронеж          308280     389300  193570  362560
Москва-Казань           835560     551240  550540  847950

общая выручка (угловая ячейка): 21 927 600


In [77]:
# Выручка по сегментам и месяцам
print(pd.pivot_table(full, index="segment", columns="month",
                     values="cost_rub", aggfunc="sum", fill_value=0))
print()

# Две функции сразу
print(pd.pivot_table(full, index="carrier", values="cost_rub",
                     aggfunc=["count", "sum", "mean"]).round(0))

month                 4        5
segment                         
Агро            1701230  1951950
Лес             1869640  1623920
Промышленность  1517100  2345630
Ритейл          1913390  1761590
Строительство   1893310  1878920
Энергетика      1701680  1769240

                   count      sum     mean
                cost_rub cost_rub cost_rub
carrier                                   
АвтоДорСервис        119  4519180  37976.0
ВолгаТранс            92  3708670  40312.0
НовыйПеревозчик        1    97050  97050.0
РусКарго              96  4096230  42669.0
СибПуть              102  4477860  43901.0
ТрансЛогистик        120  5028610  41905.0


In [78]:
# crosstab: просто количество рейсов на пересечении
print("Число рейсов: перевозчик x месяц")
print(pd.crosstab(full["carrier"], full["month"], margins=True, margins_name="Итого"))
print()

# normalize превращает количество в доли
print("Доли по строкам:")
print(pd.crosstab(full["carrier"], full["month"], normalize="index").round(3))

Число рейсов: перевозчик x месяц
month              4    5  Итого
carrier                         
АвтоДорСервис     53   66    119
ВолгаТранс        39   53     92
НовыйПеревозчик    0    1      1
РусКарго          55   41     96
СибПуть           46   56    102
ТрансЛогистик     57   63    120
Итого            250  280    530

Доли по строкам:
month                4      5
carrier                      
АвтоДорСервис    0.445  0.555
ВолгаТранс       0.424  0.576
НовыйПеревозчик  0.000  1.000
РусКарго         0.573  0.427
СибПуть          0.451  0.549
ТрансЛогистик    0.475  0.525


### Задание 5

1. Постройте сводную: выручка по **перевозчикам** (строки) и **месяцам** (столбцы),
   с итогами.
2. Постройте сводную: **средняя задержка** по маршрутам и типам груза. На пересечении
   каких маршрута и груза задержки самые большие?
3. Через `crosstab` посчитайте число рейсов по парам «сегмент клиента — тип груза».
4. Постройте сводную по менеджерам (`manager`): число рейсов и выручка. Кто ведёт
   больше всего выручки?
5. Постройте сводную **средняя задержка по перевозчикам и типам груза**. В ней есть
   и пустые ячейки, и ячейки со значением `0.00`. Чем они отличаются по смыслу?
   Как проверить догадку? И почему здесь **нельзя** писать `fill_value=0`?

In [ ]:
# 1)
p1 = pd.pivot_table(full, index="carrier", columns="month", values="cost_rub",
                    aggfunc="sum", fill_value=0, margins=True, margins_name="Итого")
print(p1)
print()

# 2)
p2 = pd.pivot_table(full, index="route", columns="cargo_type",
                    values="delay_h", aggfunc="mean").round(2)
print(p2)
print()
print("макс задержка - ", p2.stack().idxmax(), "->", p2.stack().max())
print()

# 3)
print(pd.crosstab(full["segment"], full["cargo_type"], margins=True, margins_name="Итого"))
print()


# 4)
p4 = pd.pivot_table(full, index="manager", values=["trip_id", "cost_rub"],
                    aggfunc={"trip_id": "count", "cost_rub": "sum"})
print(p4)
print("больше всего выручки - ", p4.index[0])
print()

# 5)
p5 = pd.pivot_table(full, index="carrier", columns="cargo_type",
                    values="delay_h", aggfunc="mean").round(2)
print(p5)
print()
# проверить можно, посчитав, сколько рейсов попало в каждую ячейку.
# отличаются по смыслу тем, что пустые значит нет данных, а ноль значит нет задержка
# нельзя писать, потому что он превратит нет данных в задержка 0 часов. 

month                   4         5     Итого
carrier                                      
АвтоДорСервис     2045080   2474100   4519180
ВолгаТранс        1721460   1987210   3708670
НовыйПеревозчик         0     97050     97050
РусКарго          2512120   1584110   4096230
СибПуть           1950630   2527230   4477860
ТрансЛогистик     2367060   2661550   5028610
Итого            10596350  11331250  21927600

cargo_type                    Зерно  Контейнер   Лес  Металл  Нефтепродукты  \
route                                                                         
Екатеринбург-Челябинск         1.17       1.09  0.38    0.55           1.00   
Казань-Самара                  1.00       2.70  1.54    1.50           0.77   
Москва-Воронеж                 0.00       2.00  0.89    0.40           0.77   
Москва-Казань                  2.07       0.43  1.83    0.57           3.14   
Москва-Нижний Новгород         1.00       0.92  0.53    0.57           0.58   
Москва-Санкт-Петербург         0

---
# Часть 6. Мини-проект: план/факт по маршрутам

Руководитель просит таблицу выполнения плана за май: по каждому маршруту — план, факт
и процент выполнения.

Это полный цикл сегодняшней пары: отфильтровать период → сгруппировать факт →
присоединить план → посчитать показатели → отсортировать и показать.

### Задание 6

1. Прочитайте `data/plan.csv`.
2. Посчитайте факт за май по маршрутам: число рейсов и выручку.
3. Присоедините план к факту. Какой `how` выбрать, чтобы в отчёте оказались **все**
   маршруты из плана, даже если по ним не было ни одного рейса? Проверьте число строк.
4. Посчитайте `trips_done_pct` и `revenue_done_pct` — выполнение плана в процентах.
5. Отсортируйте по выполнению плана по выручке, выведите аккуратную таблицу.
6. Посчитайте итог: суммарный план, суммарный факт, общее выполнение в процентах.

In [88]:
plan = pd.read_csv("data/plan.csv")
print(plan)

# 2)
fact = (trips[trips["month"] == 5]
        .groupby("route")
        .agg(fact_trips=("trip_id", "count"), fact_revenue=("cost_rub", "sum"))
        .reset_index()
)

# 3)
pf = plan.merge(fact, on="route", how="left", validate="1:1")
print("строк в плане:", len(plan), "| строк после соединения:", len(pf))

# 4)
pf["trips_done_pct"] = (pf["fact_trips"] / pf["plan_trips"] * 100).round(1)
pf["revenue_done_pct"] = (pf["fact_revenue"] / pf["plan_revenue"] * 100).round(1)

# 5)
pf = pf.sort_values("revenue_done_pct", ascending=False)
print(pf.to_string(index=False))

# 6)
total_plan = pf["plan_revenue"].sum()
total_fact = pf["fact_revenue"].sum()
print(f"план: {total_plan:,.0f} | факт: {total_fact:,.0f} | выполнение: {total_fact / total_plan * 100:.1f}%"
      .replace(",", " "))
print("маршруты с рейсами, но без плана - ", set(fact["route"]) - set(plan["route"]))

                          route    month  plan_trips  plan_revenue
0                 Москва-Казань  2025-05          42       1853000
1        Москва-Санкт-Петербург  2025-05          36       2591000
2        Москва-Нижний Новгород  2025-05          43       1435000
3                Москва-Воронеж  2025-05          26        954000
4                 Казань-Самара  2025-05          37        969000
5        Екатеринбург-Челябинск  2025-05          38        750000
6              Новосибирск-Омск  2025-05          24       1289000
7  Санкт-Петербург-Петрозаводск  2025-05          35       1040000
строк в плане: 8 | строк после соединения: 8
                       route   month  plan_trips  plan_revenue  fact_trips  fact_revenue  trips_done_pct  revenue_done_pct
               Москва-Казань 2025-05          42       1853000          36       2165370            85.7             116.9
Санкт-Петербург-Петрозаводск 2025-05          35       1040000          34       1203870            97.1  

### Задание 7. Выводы для руководителя

Ответьте по своей таблице:

1. Сколько маршрутов выполнили план по выручке? Сколько не выполнили?
2. Найдите маршрут, где **рейсов сделано больше плана, а выручка ниже плана**.
   Как такое возможно? Что это говорит о структуре перевозок на этом маршруте?
3. Общее выполнение плана по компании — выше или ниже 100%?
4. Напишите три предложения для руководителя: что с планом, где проблема, что проверить.

> **Выводы:**
>
> 1. План по выручке выполнили 6 маршрутов из 8. Не выполнили 2
> 2. Это Москва–Санкт-Петербург: рейсов 37 при плане 36 (102.8%), а выручка 2 351 150 при плане 2 591 000 (90.7%)
> 3. Общее выполнение плана чуть выше 100%, а именно 104.1%

План мая по выручке перевыполнен: 104.1% (11.33 млн руб. при плане 10.88 млн), его выполнили 6 маршрутов из 8.
Проблема на маршруте Москва–Санкт-Петербург: рейсов сделано больше плана (37 из 36), но выручка ниже плана
на 240 тыс. руб. (90.7%), потому что средний рейс принёс примерно на 12% меньше запланированного.
Нужно проверить, какие грузы, в каком объёме и по каким тарифам возили на Москва–Санкт-Петербург
и Москва–Воронеж (93.2%), чтобы понять, почему падает средняя стоимость рейса

---
## Отправляем работу

```
# терминал
git add .
git commit -m "Практика 9: объединение и агрегирование, план/факт по маршрутам"
git push
```

## Чек-лист сдачи

- [ ] Задание 1: таблицы склеены, число строк проверено, показан эффект несовпадения столбцов
- [ ] Задание 2: сортировки, фильтры двумя способами, цепочка с `assign`
- [ ] Задание 3: именованная агрегация, группировка по двум ключам, `transform`
- [ ] Задание 4: `merge` с проверкой числа строк, разобраны `left_only` и `right_only`
- [ ] Задание 5: пять сводных таблиц, объяснены пустые ячейки
- [ ] Задание 6: таблица план/факт с процентами выполнения
- [ ] Задание 7: письменные выводы
- [ ] Всё запушено в репозиторий

## Что унести с пары

`groupby` и `pivot_table` — это удобство: то же самое можно было посчитать циклом, просто
дольше. А вот `merge` — это риск. Он единственный из сегодняшних инструментов умеет
**менять число строк**, причём молча: никакой ошибки, просто выручка выросла на 17%.

Поэтому запомните две строки, которые должны идти за каждым соединением:

```python
assert len(result) == len(left_table)        # число строк не изменилось
print(result["key_column"].isna().sum())     # сколько не нашло пары
```

Если число строк изменилось, а вы этого не ждали — ключ в правой таблице не уникален.
Сначала разберитесь с этим, потом считайте суммы.